# Bose and Einstein

### How Planck's Law Was Rebuilt Out of Pure Counting

In 1900 Max Planck found the blackbody formula that carries his name, but he got
there by a route that troubled him for the rest of his life. He treated the walls
of the cavity as a collection of classical oscillators, borrowed Boltzmann's
statistical mechanics, and inserted the quantum of energy $\varepsilon = h\nu$ as
a bookkeeping device to make a sum come out finite. Radiation itself stayed
classical. Einstein's 1905 light quantum did not fix this: for twenty years
nobody could derive Planck's law from the photon picture alone.

In June 1924 **Satyendra Nath Bose**, a lecturer at the University of Dhaka, sent
Einstein a four-page manuscript titled *"Planck's Law and the Light Quantum
Hypothesis"* after it had been rejected by the *Philosophical Magazine*. Bose
derived Planck's law with **no classical electrodynamics and no classical
statistical mechanics anywhere in the argument**. He counted photon states in
phase space, counted the ways indistinguishable photons could be dropped into
those states, took the logarithm, and maximized. That was all.

Einstein saw immediately what he had. He translated the paper into German
himself, sent it to *Zeitschrift für Physik* with a note that it represented
"an important advance", and it appeared within weeks. He then applied the same
counting to a gas of **massive atoms**, where particle number is conserved, and
found that below a critical temperature the atoms pile into the single lowest
state. That prediction, made in 1925, was confirmed in a laboratory in 1995.

One historical note worth getting right: the paper is Bose's alone. It is
usually cited as "Bose-Einstein" because Einstein translated it, published it,
and generalized it, not because he co-wrote it.

### What makes the argument radical

Classical statistics labels the particles. Photon 1 in state A with photon 2 in
state B is a *different* arrangement from photon 2 in state A with photon 1 in
state B. Bose refused to make that distinction. He recorded only **how many**
photons sit in each state, never **which** ones. That single change in counting
is the whole difference between the Rayleigh-Jeans law and Planck's law.

This notebook walks the fourteen steps of the argument, checking each one
numerically as we go:

| Steps | What happens |
| --- | --- |
| 1-2 | Light becomes a gas of particles, and phase space is chopped into cells of volume $h^3$ |
| 3-6 | Counting the cells gives the density of states $g(\nu)\,d\nu = \dfrac{8\pi V \nu^2}{c^3}\,d\nu$ |
| 7-8 | Counting indistinguishable occupations gives $W = \dfrac{(N+g-1)!}{N!\,(g-1)!}$ |
| 9-12 | $S = k \ln W$, maximize at fixed energy, and out drops $\overline{n} = \dfrac{1}{e^{h\nu/kT}-1}$ |
| 13-14 | Multiply energy per state by number of states, and Planck's law appears |

---
## Step 1. Treat light as a gas of photons

Start by throwing away waves entirely. A cavity of volume $V$ holds a gas of
particles. Each photon of frequency $\nu$ carries

$$\varepsilon = h\nu, \qquad p = \frac{h\nu}{c},$$

the second following from the relativistic energy-momentum relation for a
massless particle, $E = pc$, so that $pc = h\nu$.

That is the entire physical input. There are no Maxwell equations below, no
standing-wave boundary conditions in the usual sense, and no oscillators in the
cavity walls. Everything else is counting.

The one number that will decide the shape of every curve in this notebook is the
ratio of the photon energy to the thermal energy,

$$x = \frac{h\nu}{kT}.$$

When $x \ll 1$ a photon is cheap and the classical answer will survive. When
$x \gg 1$ a single photon costs more than the thermal energy available, and the
mode is essentially frozen out. Planck's law is the story of that crossover.

In [ ]:
"""bose_einstein.ipynb"""

# Cell 01 - Imports, constants, and the ratio that controls everything

%matplotlib inline

import itertools
from math import lgamma

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle
from scipy.constants import (
    Boltzmann,
    Planck,
    Stefan_Boltzmann,
    Wien,
    elementary_charge,
    speed_of_light,
)
from scipy.integrate import quad
from scipy.optimize import minimize, root_scalar
from scipy.special import zeta

H_PLANCK = Planck  # J s
K_BOLTZMANN = Boltzmann  # J / K
C_LIGHT = speed_of_light  # m / s

SUN_TEMPERATURE = 5772.0  # K, the Sun's effective photosphere temperature
CMB_TEMPERATURE = 2.725  # K, the cosmic microwave background today

print(f"h  = {H_PLANCK:.6e} J s")
print(f"k  = {K_BOLTZMANN:.6e} J / K")
print(f"c  = {C_LIGHT:.6e} m / s")
print()

# x = h nu / kT is the only combination that appears in the final answer
print(
    f"{'source':<22}{'nu (Hz)':>12}{'h nu (eV)':>12}{'T (K)':>10}{'x = h nu / kT':>16}"
)
for label, frequency, temperature in [
    ("CMB peak", 1.60e11, CMB_TEMPERATURE),
    ("microwave oven", 2.45e9, 300.0),
    ("infrared, 10 um", 3.00e13, 300.0),
    ("green light", 5.45e14, SUN_TEMPERATURE),
    ("near ultraviolet", 1.00e15, SUN_TEMPERATURE),
]:
    photon_energy = H_PLANCK * frequency
    ratio = photon_energy / (K_BOLTZMANN * temperature)
    print(
        f"{label:<22}{frequency:12.3e}{photon_energy / elementary_charge:12.4f}"
        f"{temperature:10.1f}{ratio:16.4f}"
    )

In [ ]:
# Cell 02 - Plot 1: the two facts about a photon, energy and momentum

frequency = np.logspace(8, 20, 600)
photon_energy = H_PLANCK * frequency  # J
photon_momentum = H_PLANCK * frequency / C_LIGHT  # kg m / s

VISIBLE_LOW, VISIBLE_HIGH = 4.3e14, 7.5e14  # Hz

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.loglog(frequency, photon_energy / elementary_charge, color="royalblue", lw=2)
ax.axhline(
    K_BOLTZMANN * SUN_TEMPERATURE / elementary_charge,
    color="crimson",
    ls="--",
    lw=1.2,
    label=f"$kT$ at {SUN_TEMPERATURE:.0f} K (the Sun)",
)
ax.axhline(
    K_BOLTZMANN * CMB_TEMPERATURE / elementary_charge,
    color="darkorange",
    ls="--",
    lw=1.2,
    label=f"$kT$ at {CMB_TEMPERATURE} K (the CMB)",
)
ax.axvspan(VISIBLE_LOW, VISIBLE_HIGH, color="palegreen", alpha=0.6, label="visible")
ax.set_xlabel(r"frequency $\nu$ (Hz)")
ax.set_ylabel(r"photon energy $h\nu$ (eV)")
ax.set_title(r"Step 1a: every photon carries $\varepsilon = h\nu$")
ax.legend(fontsize=8, loc="upper left")
ax.grid(True, which="both", alpha=0.3)

ax = axes[1]
ax.loglog(frequency, photon_momentum, color="seagreen", lw=2)
ax.axvspan(VISIBLE_LOW, VISIBLE_HIGH, color="palegreen", alpha=0.6, label="visible")
ax.set_xlabel(r"frequency $\nu$ (Hz)")
ax.set_ylabel(r"photon momentum $p = h\nu / c$ (kg m/s)")
ax.set_title(r"Step 1b: and momentum $p = h\nu / c$")
ax.legend(fontsize=8, loc="upper left")
ax.grid(True, which="both", alpha=0.3)

fig.suptitle("A Photon Gas Needs Only Two Numbers per Particle")
fig.tight_layout()
plt.show()

# Both lines are straight on log-log with slope 1, since both are linear in nu
green = 5.45e14
print(f"a green photon at {green:.3e} Hz:")
print(
    f"  energy   = {H_PLANCK * green:.6e} J = {H_PLANCK * green / elementary_charge:.4f} eV"
)
print(f"  momentum = {H_PLANCK * green / C_LIGHT:.6e} kg m/s")
print(
    f"  p * c    = {H_PLANCK * green / C_LIGHT * C_LIGHT:.6e} J  (equals h nu, as E = pc requires)"
)

---
## Step 2. Chop phase space into cells of volume $h^3$

A classical particle needs six numbers to be fully specified: three of position
and three of momentum. Those six axes form **phase space**,

$$(x,\ y,\ z,\ p_x,\ p_y,\ p_z).$$

Bose's move was to declare that a quantum state is not a point in this space but
a **cell of finite volume**. The uncertainty principle fixes the size: each
conjugate pair contributes one factor of $h$,

$$\Delta x\,\Delta p_x \sim h, \qquad
\Delta y\,\Delta p_y \sim h, \qquad
\Delta z\,\Delta p_z \sim h,$$

so one state occupies

$$\Delta x\,\Delta y\,\Delta z\,\Delta p_x\,\Delta p_y\,\Delta p_z \sim h^3.$$

You can see the same number without invoking uncertainty at all. Put the photons
in a cube of side $L$ with periodic boundaries. A plane wave $e^{i k \cdot r}$
fits only if each component of $k$ is a multiple of $2\pi/L$, so the allowed
momenta form a cubic lattice with spacing

$$\Delta p = \frac{h}{L}.$$

Each lattice point owns a momentum-space box of volume $(h/L)^3$ and the whole
position-space volume $V = L^3$, and the product is exactly $h^3$. The cell
volume is not an approximation. It is what "one mode" means.

The plot below shows this in one dimension, where phase space is just the
$(x, p_x)$ plane.

In [ ]:
# Cell 03 - Plot 2: phase space divided into cells of area h (one dimension)

BOX_SIDE = 1.0e-6  # m, a one micron box, small enough that the modes are countable
MOMENTUM_STEP = H_PLANCK / BOX_SIDE  # spacing of the allowed momenta, h / L

n_levels = np.arange(-4, 5)
allowed_momenta = n_levels * MOMENTUM_STEP

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: the (x, p) plane cut into cells, each of area L * (h / L) = h
ax = axes[0]
for n in n_levels:
    lower = (n - 0.5) * MOMENTUM_STEP
    ax.add_patch(
        Rectangle(
            (0.0, lower / MOMENTUM_STEP),
            1.0,
            1.0,
            fc="lightsteelblue" if n % 2 == 0 else "white",
            ec="k",
            lw=0.8,
        )
    )
    ax.plot([0, 1], [n, n], color="crimson", lw=2)
    ax.plot(0.5, n, "o", color="crimson", ms=7)
ax.text(
    0.5,
    4.9,
    "each rectangle has area  $L \\times h/L = h$",
    ha="center",
    fontsize=10,
    bbox={"boxstyle": "round", "fc": "papayawhip", "ec": "k"},
)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-4.8, 5.6)
ax.set_xlabel("$x / L$")
ax.set_ylabel("$p_x$  (units of $h/L$)")
ax.set_title("One quantum state per cell of area $h$")

# Right: counting states directly against the continuum estimate 2 p L / h
momentum_cut = np.linspace(0.0, 8.5 * MOMENTUM_STEP, 400)
wide_levels = np.arange(-40, 41) * MOMENTUM_STEP
exact_wide = np.array([np.sum(np.abs(wide_levels) <= p) for p in momentum_cut])

ax = axes[1]
ax.step(
    momentum_cut / MOMENTUM_STEP,
    exact_wide,
    where="post",
    color="royalblue",
    lw=1.8,
    label="exact count of lattice points",
)
ax.plot(
    momentum_cut / MOMENTUM_STEP,
    2 * momentum_cut * BOX_SIDE / H_PLANCK,
    color="crimson",
    ls="--",
    lw=1.8,
    label=r"phase-space estimate $2 p L / h$",
)
ax.set_xlabel("$p$  (units of $h/L$)")
ax.set_ylabel("number of states with momentum below $p$")
ax.set_title("Counting cells reproduces the exact mode count")
ax.legend(fontsize=9, loc="upper left")
ax.grid(True, alpha=0.3)

fig.suptitle("Step 2: Phase Space Is Quantized into Cells of Volume $h$ per Dimension")
fig.tight_layout()
plt.show()

print(f"box side              L = {BOX_SIDE:.3e} m")
print(f"momentum spacing  h / L = {MOMENTUM_STEP:.6e} kg m/s")
print(f"cell area   L * (h / L) = {BOX_SIDE * MOMENTUM_STEP:.6e} J s")
print(f"Planck constant       h = {H_PLANCK:.6e} J s  (they are the same number)")
print()
for cut in [2.5, 5.5, 10.5]:
    counted = int(np.sum(np.abs(wide_levels) <= cut * MOMENTUM_STEP))
    print(
        f"states with |p| <= {cut:4.1f} h/L: counted {counted:3d}, "
        f"phase-space estimate {2 * cut:.1f}"
    )

---
## Steps 3, 4, and 5. Count the cells in a spherical shell

Now do the counting in three dimensions.

**Step 3.** The position part is trivial. The photons are free to be anywhere in
the cavity, so the position-space volume available is just $V$.

**Step 4.** For the momentum part, ask how many states have momentum magnitude
between $p$ and $p + dp$. All such momenta lie in a thin spherical shell in
momentum space, of volume

$$4\pi p^2\,dp.$$

The phase-space volume of the shell is $V \cdot 4\pi p^2 dp$, and dividing by the
cell volume $h^3$ gives the number of cells,

$$\frac{4\pi V p^2\,dp}{h^3}.$$

**Step 5.** One correction remains. A photon of a given momentum still has two
independent polarizations, so every cell holds two distinct states:

$$g(p)\,dp = \frac{8\pi V p^2\,dp}{h^3}.$$

The factor of 2 is the only place in the whole derivation where a property of
light beyond $\varepsilon = h\nu$ and $p = h\nu/c$ enters.

The two plots below check the counting by brute force. We build the actual
lattice of allowed momenta in a cubic box and count the points, with no
continuum approximation anywhere, then compare against the formulas.

In [ ]:
# Cell 04 - Plot 3: the lattice of allowed momenta, and the count inside a sphere

LATTICE_MAX = 8  # draw the lattice points out to |n| = 8
SHELL_INNER, SHELL_OUTER = 5.5, 6.5  # the highlighted shell, in units of h / L

axis_range = np.arange(-LATTICE_MAX, LATTICE_MAX + 1)
n_x, n_y, n_z = np.meshgrid(axis_range, axis_range, axis_range, indexing="ij")
n_x, n_y, n_z = n_x.ravel(), n_y.ravel(), n_z.ravel()
radius = np.sqrt(n_x**2 + n_y**2 + n_z**2)

inside = radius <= LATTICE_MAX
in_shell = (radius > SHELL_INNER) & (radius <= SHELL_OUTER)

fig = plt.figure(figsize=(13, 5.5))

# Left: the lattice points themselves, with one spherical shell picked out.
# Only the half with n_y >= 0 is drawn, so the shell can be seen in cross section.
ax = fig.add_subplot(1, 2, 1, projection="3d")
visible_half = inside & (n_y >= 0)
bulk = visible_half & ~in_shell
shell_half = visible_half & in_shell
ax.plot(
    n_x[bulk],
    n_y[bulk],
    n_z[bulk],
    ls="",
    marker="o",
    ms=2.6,
    color="lightsteelblue",
    alpha=0.45,
)
ax.plot(
    n_x[shell_half],
    n_y[shell_half],
    n_z[shell_half],
    ls="",
    marker="o",
    ms=4.4,
    color="crimson",
)
ax.set_box_aspect((1, 1, 1))  # without this the sphere is drawn squashed
ax.view_init(elev=18, azim=-58)
ax.set_xlabel("$n_x$")
ax.set_ylabel("$n_y$")
ax.set_zlabel("$n_z$")
ax.set_title(
    f"Allowed momenta $p = (h/L)\\,\\mathbf{{n}}$, cut open\n"
    f"red shell: {SHELL_INNER} < |n| <= {SHELL_OUTER}"
)

# Right: cumulative count of lattice points against the sphere volume
cut_radius = np.linspace(1.0, 40.0, 160)
big_axis = np.arange(-40, 41)
b_x, b_y, b_z = np.meshgrid(big_axis, big_axis, big_axis, indexing="ij")
big_radius = np.sqrt(b_x**2 + b_y**2 + b_z**2).ravel()
big_radius = big_radius[big_radius > 0]  # drop the zero mode, which carries no energy
counted = np.array([np.sum(big_radius <= r) for r in cut_radius])
continuum = (4.0 / 3.0) * np.pi * cut_radius**3

ax = fig.add_subplot(1, 2, 2)
ax.plot(cut_radius, counted, color="royalblue", lw=1.8, label="lattice points counted")
ax.plot(
    cut_radius,
    continuum,
    color="crimson",
    ls="--",
    lw=1.8,
    label=r"sphere volume $\frac{4}{3}\pi n^3$",
)
ax.set_xlabel("$|n| = pL/h$")
ax.set_ylabel("states with momentum below $p$ (one polarization)")
ax.set_yscale("log")
ax.set_xscale("log")
ax.legend(fontsize=9, loc="upper left")
ax.grid(True, which="both", alpha=0.3)
ax.set_title("Counting cells and integrating agree once $|n|$ is large")

fig.suptitle(
    "Steps 3-4: The Momentum States Form a Lattice, and a Shell Holds "
    r"$4\pi p^2 dp \cdot V / h^3$ of Them"
)
fig.tight_layout()
plt.show()

print(f"{'|n|':>6}{'counted':>12}{'4/3 pi n^3':>14}{'relative error':>16}")
for r in [5.0, 10.0, 20.0, 40.0]:
    exact = int(np.sum(big_radius <= r))
    smooth = (4.0 / 3.0) * np.pi * r**3
    print(f"{r:6.1f}{exact:12d}{smooth:14.1f}{exact / smooth - 1:16.4%}")
print()
print(f"photons in the red shell (one polarization) = {int(np.sum(in_shell))}")
print(
    f"same shell from 4 pi n^2 dn                 = "
    f"{4 * np.pi * ((SHELL_INNER + SHELL_OUTER) / 2) ** 2 * (SHELL_OUTER - SHELL_INNER):.1f}"
)

In [ ]:
# Cell 05 - Plot 4: how many states live in each momentum shell

SHELL_BOX = 5.0e-6  # m, box side used for the shell census
SHELL_MAX = 30  # count lattice points out to |n| = 30

shell_axis = np.arange(-SHELL_MAX, SHELL_MAX + 1)
s_x, s_y, s_z = np.meshgrid(shell_axis, shell_axis, shell_axis, indexing="ij")
shell_radius = np.sqrt(s_x**2 + s_y**2 + s_z**2).ravel()
shell_radius = shell_radius[(shell_radius > 0) & (shell_radius <= SHELL_MAX)]

# Convert lattice index to a physical momentum, p = (h / L) |n|
lattice_momentum = shell_radius * H_PLANCK / SHELL_BOX
box_volume = SHELL_BOX**3

edges = np.linspace(0.0, SHELL_MAX * H_PLANCK / SHELL_BOX, 31)
centers = 0.5 * (edges[:-1] + edges[1:])
width = edges[1] - edges[0]
# 2 polarizations per lattice point, then per unit momentum
counts, _ = np.histogram(lattice_momentum, bins=edges)
density_counted = 2 * counts / width
density_formula = 8 * np.pi * box_volume * centers**2 / H_PLANCK**3

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.bar(
    centers,
    density_counted,
    width=width * 0.9,
    color="lightsteelblue",
    ec="steelblue",
    label="counted from the lattice (2 polarizations)",
)
ax.plot(
    centers,
    density_formula,
    color="crimson",
    lw=2,
    marker="o",
    ms=4,
    label=r"$g(p) = 8\pi V p^2 / h^3$",
)
ax.set_xlabel("$p$ (kg m/s)")
ax.set_ylabel("states per unit momentum")
ax.legend(fontsize=9, loc="upper left")
ax.grid(True, alpha=0.3)
ax.set_title(
    f"Step 5: Density of Photon States in a Box of Side {SHELL_BOX * 1e6:.0f} $\\mu$m"
)
fig.tight_layout()
plt.show()

total_counted = 2 * len(shell_radius)
p_max = SHELL_MAX * H_PLANCK / SHELL_BOX
total_formula = 8 * np.pi * box_volume * p_max**3 / (3 * H_PLANCK**3)
print(f"states counted below p_max = {total_counted}")
print(f"states from 8 pi V p^3 / 3 h^3 = {total_formula:.1f}")
print(f"agreement to {abs(total_counted / total_formula - 1):.3%}")

---
## Step 6. Trade momentum for frequency

The last step left the answer in terms of momentum, but a spectrum is measured in
frequency. For a photon the two are proportional,

$$p = \frac{h\nu}{c}, \qquad dp = \frac{h}{c}\,d\nu,$$

so substituting into $g(p)\,dp = 8\pi V p^2\,dp / h^3$ gives

$$g(\nu)\,d\nu = \frac{8\pi V}{h^3}\left(\frac{h^2\nu^2}{c^2}\right)\left(\frac{h}{c}\,d\nu\right).$$

Every factor of $h$ cancels, $h^2 \cdot h / h^3 = 1$, and we are left with

$$\boxed{\;g(\nu)\,d\nu = \frac{8\pi V \nu^2}{c^3}\,d\nu\;}
\qquad\text{or per unit volume}\qquad
\frac{g(\nu)}{V} = \frac{8\pi\nu^2}{c^3}.$$

Two things are worth pausing over.

First, **Planck's constant has vanished**. The density of states is a purely
geometric result. This is the same $8\pi\nu^2/c^3$ that Rayleigh and Jeans got by
counting classical standing waves, which is reassuring: the counting of modes was
never the problem with classical physics.

Second, **this is not yet Planck's law**. It says how many boxes there are, not
how much energy is in each. Everything quantum is still ahead of us, in how the
photons are distributed among these states.

In [ ]:
# Cell 06 - Plot 5: the density of states in frequency, counted and predicted


def density_of_states(frequency: float | np.ndarray) -> np.ndarray:
    """Return the number of photon states per unit volume per unit frequency.

    Parameters
    ----------
    frequency : float | np.ndarray
        Frequency in Hz.

    Returns
    -------
    np.ndarray
        The value of 8 pi nu^2 / c^3, in states per cubic meter per hertz.
    """
    nu = np.asarray(frequency, dtype=float)
    return 8.0 * np.pi * nu**2 / C_LIGHT**3


# Reuse the lattice from the previous cell: nu = c |n| / L for a periodic box
lattice_frequency = shell_radius * C_LIGHT / SHELL_BOX
freq_edges = np.linspace(0.0, lattice_frequency.max(), 31)
freq_centers = 0.5 * (freq_edges[:-1] + freq_edges[1:])
freq_width = freq_edges[1] - freq_edges[0]
freq_counts, _ = np.histogram(lattice_frequency, bins=freq_edges)
counted_density = 2 * freq_counts / (freq_width * box_volume)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.bar(
    freq_centers / 1e12,
    counted_density,
    width=freq_width * 0.9 / 1e12,
    color="lightsteelblue",
    ec="steelblue",
    label="counted from the lattice",
)
ax.plot(
    freq_centers / 1e12,
    density_of_states(freq_centers),
    color="crimson",
    lw=2,
    marker="o",
    ms=4,
    label=r"$8\pi\nu^2/c^3$",
)
ax.set_xlabel(r"$\nu$ (THz)")
ax.set_ylabel(r"states m$^{-3}$ Hz$^{-1}$")
ax.legend(fontsize=9, loc="upper left")
ax.grid(True, alpha=0.3)
ax.set_title("The count is quadratic in frequency")

ax = axes[1]
wide_frequency = np.logspace(11, 16, 400)
ax.loglog(wide_frequency, density_of_states(wide_frequency), color="royalblue", lw=2)
ax.axvspan(VISIBLE_LOW, VISIBLE_HIGH, color="palegreen", alpha=0.6, label="visible")
ax.set_xlabel(r"$\nu$ (Hz)")
ax.set_ylabel(r"$g(\nu)/V$  (states m$^{-3}$ Hz$^{-1}$)")
ax.legend(fontsize=9, loc="upper left")
ax.grid(True, which="both", alpha=0.3)
ax.set_title(r"Slope 2 forever: nothing here cuts off the high frequencies")

fig.suptitle("Step 6: The Density of States Contains No Planck Constant at All")
fig.tight_layout()
plt.show()

test_frequency = 3.0e14
print(f"at nu = {test_frequency:.2e} Hz")
print(
    f"  g(nu)/V from the formula = {density_of_states(test_frequency):.6e} per m^3 per Hz"
)
print()
print("counted vs formula, sampled across the range:")
print(f"{'nu (THz)':>12}{'counted':>16}{'formula':>16}{'ratio':>10}")
for index in [5, 10, 15, 20, 25, 29]:
    center = freq_centers[index]
    counted = counted_density[index]
    predicted = density_of_states(center)
    print(
        f"{center / 1e12:12.2f}{counted:16.4e}{predicted:16.4e}{counted / predicted:10.4f}"
    )
print()
print("The lowest bins wander a few percent either way because a small sphere")
print("holds only a handful of lattice points. The formula is the smooth limit.")

---
## Steps 7 and 8. Count the ways to distribute indistinguishable photons

Here is the heart of the paper.

Take a narrow frequency interval containing $g$ states and $N$ photons. In how
many ways can the photons be arranged?

Bose's answer: a configuration is nothing but a list of **occupation numbers**.
With three states and two photons, $(2,0,0)$ means both photons sit in the first
state, and $(1,1,0)$ means one in each of the first two. Photons may share a
state freely, since there is no exclusion principle for them.

Counting such lists is the classic **stars and bars** problem. Draw $N$ stars for
the photons and $g-1$ bars to separate them into $g$ states. Every distinct
sequence of stars and bars is one configuration, so

$$\boxed{\;W = \frac{(N+g-1)!}{N!\,(g-1)!}\;}$$

and for large $g$ this is written $W \approx (N+g)! / (N!\,g!)$.

### Why this is revolutionary

Classical Maxwell-Boltzmann statistics labels the particles: photon 1, photon 2,
photon 3. Swapping two labeled photons between states counts as a **new**
arrangement, and the number of arrangements is $g^N$.

Bose never wrote the labels down. In his counting the configuration $(2,1,0)$ is
one configuration, full stop. Exchanging the two photons in the first state
changes nothing, because there is nothing to exchange: the photons have no
identity to swap.

For contrast, fermions obey the opposite extreme, at most one particle per state,
which gives $W = \binom{g}{N}$. The three counting rules diverge dramatically as
the gas gets crowded, which is exactly the regime where Planck's law departs from
the classical one.

In [ ]:
# Cell 07 - Plot 6: every way to arrange N indistinguishable photons in g states


def occupation_patterns(n_photons: int, n_states: int) -> list[tuple[int, ...]]:
    """Return every occupation-number list for indistinguishable particles.

    Uses the stars and bars construction: choose the positions of the
    n_states - 1 bars among n_photons + n_states - 1 slots, and read the runs
    of stars between them as the occupation numbers.

    Parameters
    ----------
    n_photons : int
        Number of indistinguishable photons, the "stars".
    n_states : int
        Number of available quantum states, separated by "bars".

    Returns
    -------
    list[tuple[int, ...]]
        One tuple of length n_states per distinct configuration.
    """
    patterns: list[tuple[int, ...]] = []
    slots = n_photons + n_states - 1
    for bars in itertools.combinations(range(slots), n_states - 1):
        counts: list[int] = []
        previous = -1
        for bar in bars:
            counts.append(bar - previous - 1)
            previous = bar
        counts.append(slots - previous - 1)
        patterns.append(tuple(counts))
    return patterns


def bose_count(n_photons: int, n_states: int) -> float:
    """Return the Bose number of arrangements, using log-gamma to stay finite."""
    return np.exp(
        lgamma(n_photons + n_states) - lgamma(n_photons + 1) - lgamma(n_states)
    )


DEMO_PHOTONS, DEMO_STATES = 3, 3
patterns = occupation_patterns(DEMO_PHOTONS, DEMO_STATES)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# Left: draw all of the configurations as photons sitting in boxes
ax = axes[0]
for row, pattern in enumerate(patterns):
    y = -row
    for state, occupancy in enumerate(pattern):
        ax.add_patch(Rectangle((state, y - 0.4), 0.9, 0.8, fc="white", ec="k", lw=0.8))
        for photon in range(occupancy):
            ax.plot(
                0.45 + state,
                y - 0.22 + 0.22 * photon,
                "o",
                color="crimson",
                ms=7,
            )
    ax.text(DEMO_STATES + 0.15, y, str(pattern), va="center", fontsize=10)
ax.set_xlim(-0.3, DEMO_STATES + 1.4)
ax.set_ylim(-len(patterns) + 0.3, 1.1)
ax.set_xticks([])
ax.set_yticks([])
ax.set_title(
    f"All {len(patterns)} configurations of "
    f"{DEMO_PHOTONS} photons in {DEMO_STATES} states"
)

# Right: how the three counting rules diverge as the gas gets crowded
ax = axes[1]
STATES = 20
photon_numbers = list(range(1, 21))
bose = np.array([bose_count(n, STATES) for n in photon_numbers])
boltzmann = np.array([float(STATES) ** n for n in photon_numbers])
fermi = np.array(
    [
        np.exp(lgamma(STATES + 1) - lgamma(n + 1) - lgamma(STATES - n + 1))
        for n in photon_numbers
    ]
)

ax.semilogy(
    photon_numbers,
    boltzmann,
    "s-",
    color="darkorange",
    label=r"labeled particles: $g^N$",
)
ax.semilogy(
    photon_numbers,
    bose,
    "o-",
    color="royalblue",
    label=r"Bose: $\frac{(N+g-1)!}{N!(g-1)!}$",
)
ax.semilogy(
    photon_numbers, fermi, "^-", color="seagreen", label=r"Fermi: $\binom{g}{N}$"
)
ax.set_xlabel("photons $N$")
ax.set_ylabel("number of arrangements $W$")
ax.set_title(f"Three ways to count, with $g$ = {STATES} states")
ax.legend(fontsize=10, loc="upper left")
ax.grid(True, which="both", alpha=0.3)

fig.suptitle("Steps 7-8: Counting Occupations, Not Particles")
fig.tight_layout()
plt.show()

print(
    f"enumerated configurations for N = {DEMO_PHOTONS}, g = {DEMO_STATES}: {len(patterns)}"
)
print(
    f"stars and bars formula                     : "
    f"{bose_count(DEMO_PHOTONS, DEMO_STATES):.0f}"
)
print(f"the classic example, N = 2 and g = 3       : {occupation_patterns(2, 3)}")
print()
print(
    f"at N = g = 20, labeled counting overshoots Bose by a factor of "
    f"{boltzmann[-1] / bose[-1]:.3e}"
)

---
## Step 9. Turn arrangements into entropy

Boltzmann's tombstone supplies the bridge from counting to thermodynamics,

$$S = k \ln W.$$

The frequency intervals are independent, so the arrangements multiply,
$W_\text{total} = \prod_i W_i$, and the logarithm turns the product into a sum:

$$S = k \sum_i \ln W_i
    = k \sum_i \left[\ln (N_i+g_i)! - \ln N_i! - \ln g_i!\right].$$

Factorials of astronomical numbers are useless as they stand, so apply
**Stirling's approximation**, $\ln n! \approx n \ln n - n$. The linear terms
cancel between the three factorials, since
$(N+g) - N - g = 0$, leaving a clean result:

$$\boxed{\;S = k\sum_i\left[(N_i+g_i)\ln(N_i+g_i) - N_i \ln N_i - g_i \ln g_i\right]\;}$$

Divide through by $g_i$ and write $n_i = N_i/g_i$ for the average occupancy of a
state, and the entropy per state takes a form that depends on nothing but $n$:

$$\frac{S_i}{k\,g_i} = (1+n_i)\ln(1+n_i) - n_i \ln n_i.$$

This function is the entire thermodynamic content of Bose statistics. Note its
shape in the plot below: it grows without bound as $n$ increases (you can always
add another photon), but only logarithmically. That slow growth is what will
eventually stop the ultraviolet catastrophe.

In [ ]:
# Cell 08 - Plot 7: Stirling's approximation and the entropy per state


def entropy_per_state(occupancy: float | np.ndarray) -> float | np.ndarray:
    """Return S / (k g) for a Bose state with mean occupancy n."""
    n = np.asarray(occupancy, dtype=float)
    return (1.0 + n) * np.log1p(n) - np.where(
        n > 0, n * np.log(np.where(n > 0, n, 1)), 0.0
    )


fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: how quickly Stirling's approximation becomes good enough
ax = axes[0]
n_values = np.arange(2, 201)
exact_log_factorial = np.array([lgamma(n + 1) for n in n_values])
stirling = n_values * np.log(n_values) - n_values
ax.semilogy(
    n_values,
    np.abs(stirling / exact_log_factorial - 1.0),
    color="royalblue",
    lw=2,
)
ax.set_xlabel("$n$")
ax.set_ylabel(r"relative error of $\ln n! \approx n\ln n - n$")
ax.grid(True, which="both", alpha=0.3)
ax.set_title("Stirling's approximation, error versus $n$")

# Right: the entropy of one state as a function of how crowded it is
ax = axes[1]
occupancy = np.logspace(-3, 2, 400)
ax.semilogx(
    occupancy,
    entropy_per_state(occupancy),
    color="crimson",
    lw=2,
    label=r"$(1+n)\ln(1+n) - n\ln n$",
)
ax.semilogx(
    occupancy,
    np.log(occupancy) + 1.0,
    color="gray",
    ls="--",
    lw=1.4,
    label=r"crowded limit $\ln n + 1$",
)
ax.set_xlabel("mean occupancy $n = N/g$")
ax.set_ylabel("entropy per state  $S / (k g)$")
ax.set_ylim(-0.2, 6.0)
ax.legend(fontsize=9, loc="upper left")
ax.grid(True, which="both", alpha=0.3)
ax.set_title("Entropy of a single Bose state")

fig.suptitle("Step 9: From Counting to Entropy, by Way of Stirling")
fig.tight_layout()
plt.show()

# Check the Stirling form of S against the exact log of the stars-and-bars count
print(f"{'N':>8}{'g':>8}{'exact ln W':>16}{'Stirling S/k':>16}{'ratio':>10}")
for n_photons, n_states in [(10, 10), (100, 100), (1000, 1000), (100000, 100000)]:
    exact = lgamma(n_photons + n_states) - lgamma(n_photons + 1) - lgamma(n_states)
    approx = (
        (n_photons + n_states) * np.log(n_photons + n_states)
        - n_photons * np.log(n_photons)
        - n_states * np.log(n_states)
    )
    print(
        f"{n_photons:8d}{n_states:8d}{exact:16.4f}{approx:16.4f}{approx / exact:10.5f}"
    )

---
## Steps 10 and 11. Maximize the entropy at fixed energy

The photons are free to shuffle between frequency intervals, but the cavity holds
a fixed total energy,

$$E = \sum_i N_i\,h\nu_i.$$

Thermal equilibrium is the distribution that makes the entropy as large as it can
be **subject to that one constraint**. Introduce a Lagrange multiplier $\lambda$
and maximize $S - \lambda E$, which requires

$$\frac{\partial S}{\partial N_i} - \lambda h\nu_i = 0 \quad\text{for every } i.$$

Differentiating the entropy from step 9, and remembering that $g_i$ is fixed,

$$\frac{\partial S_i}{\partial N_i}
  = k\left[\ln(N_i+g_i) - \ln N_i\right]
  = k \ln\!\left(\frac{N_i+g_i}{N_i}\right),$$

so equilibrium demands

$$k \ln\!\left(\frac{N_i+g_i}{N_i}\right) = \lambda h\nu_i.$$

Thermodynamics identifies the multiplier as $\lambda = 1/(kT)$, since
$\partial S/\partial E = 1/T$. Then

$$\ln\!\left(\frac{N_i+g_i}{N_i}\right) = \frac{h\nu_i}{kT}
\;\Longrightarrow\;
1 + \frac{g_i}{N_i} = e^{h\nu_i/kT}
\;\Longrightarrow\;
\frac{g_i}{N_i} = e^{h\nu_i/kT} - 1,$$

and inverting gives the **Bose-Einstein occupation factor**:

$$\boxed{\;\overline{n}(\nu) = \frac{N_i}{g_i} = \frac{1}{e^{h\nu/kT}-1}\;}$$

The plots check this two ways. On the left we simply plot the quantity being
maximized for a single frequency and look at where its peak falls. On the right
we hand a numerical optimizer a dozen frequency bins, tell it only the total
energy, and let it find the entropy maximum on its own with no formula supplied.

In [ ]:
# Cell 09 - Plot 8: maximizing S - lambda E, by eye and numerically


def bose_occupancy(x: float | np.ndarray) -> float | np.ndarray:
    """Return the mean photon occupancy of a state with x = h nu / kT."""
    return 1.0 / np.expm1(np.asarray(x, dtype=float))


fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: the objective S/k - lambda E for one frequency bin, in units of kT
ax = axes[0]
occupancy = np.logspace(-2, 1.4, 500)
for x_value, color in [(0.5, "royalblue"), (1.0, "seagreen"), (2.0, "crimson")]:
    objective = entropy_per_state(occupancy) - x_value * occupancy
    ax.semilogx(
        occupancy, objective, color=color, lw=2, label=f"$h\\nu/kT$ = {x_value}"
    )
    peak = bose_occupancy(x_value)
    ax.plot(
        peak,
        entropy_per_state(peak) - x_value * peak,
        "o",
        color=color,
        ms=9,
        mec="k",
        zorder=5,
    )
    ax.axvline(peak, color=color, ls=":", lw=1)
ax.set_xlabel("occupancy $n$")
ax.set_ylabel(r"$\left(S - \lambda E\right) / (k g)$")
ax.set_ylim(-1.6, 1.4)  # the curves plunge far below this, but the peaks are the point
ax.set_title(r"Markers sit at $n = 1/(e^{h\nu/kT}-1)$")
ax.legend(fontsize=9, loc="lower left")
ax.grid(True, which="both", alpha=0.3)

# Right: let an optimizer find the maximum with only the energy constraint
BIN_COUNT = 12
x_bins = np.linspace(0.4, 8.0, BIN_COUNT)  # bin energies in units of kT
g_bins = x_bins**2  # degeneracies follow the density of states, g ~ nu^2

target_occupancy = bose_occupancy(x_bins)
target_energy = float(np.sum(g_bins * target_occupancy * x_bins))


def negative_entropy(occupancies: np.ndarray) -> float:
    """Return -S/k for a trial set of occupancies (the optimizer minimizes)."""
    return -float(np.sum(g_bins * entropy_per_state(occupancies)))


def energy_residual(occupancies: np.ndarray) -> float:
    """Return the amount by which the trial energy misses the fixed total."""
    return float(np.sum(g_bins * occupancies * x_bins)) - target_energy


solution = minimize(
    negative_entropy,
    x0=np.full(BIN_COUNT, 0.5),
    method="SLSQP",
    bounds=[(1e-9, None)] * BIN_COUNT,
    constraints=[{"type": "eq", "fun": energy_residual}],
    options={"maxiter": 500, "ftol": 1e-14},
)
numeric_occupancy = solution.x

ax = axes[1]
smooth_x = np.linspace(0.3, 8.5, 300)
ax.semilogy(
    smooth_x,
    bose_occupancy(smooth_x),
    color="crimson",
    lw=2,
    label=r"$1/(e^{h\nu/kT}-1)$, derived above",
)
ax.semilogy(
    x_bins,
    numeric_occupancy,
    "o",
    color="royalblue",
    ms=9,
    mec="k",
    label="entropy maximized numerically",
)
ax.set_xlabel(r"$x = h\nu / kT$")
ax.set_ylabel("mean occupancy $n$")
ax.set_title("The optimizer was given only the total energy")
ax.legend(fontsize=9, loc="upper right")
ax.grid(True, which="both", alpha=0.3)

fig.suptitle("Steps 10-11: Equilibrium Is Whatever Maximizes the Entropy")
fig.tight_layout()
plt.show()

print(f"optimizer converged: {solution.success}  ({solution.message})")
print(
    f"largest deviation from the Bose formula = "
    f"{np.max(np.abs(numeric_occupancy / target_occupancy - 1)):.3e}"
)
print()
# The Lagrange multiplier can be read back out of each bin, and must be the same
recovered = np.log(1.0 + 1.0 / numeric_occupancy) / x_bins
print("lambda recovered bin by bin (in units of 1/kT), should all be 1.0:")
print(np.round(recovered, 6))

---
## Step 12. Why no chemical potential appears

For an ordinary gas of atoms, the particles are conserved. Maximizing the entropy
then requires **two** Lagrange multipliers, one for the energy and one for the
particle number, and the second shows up in the answer as the chemical potential
$\mu$:

$$\overline{n}(\varepsilon) = \frac{1}{e^{(\varepsilon-\mu)/kT}-1}.$$

Photons are different. The walls of the cavity absorb and emit them constantly,

$$\text{wall energy} \rightleftharpoons \text{photons},$$

so the number of photons in the cavity is **not** a conserved quantity. There is
no second constraint to impose, no second multiplier to introduce, and therefore

$$\mu = 0.$$

This is not a convenient simplification, it is a physical statement: the photon
gas has no independent control over its particle number. Fix the temperature and
the volume and the photon number is fixed too, with nothing left to adjust. The
right-hand plot below computes it, and finds

$$\frac{N}{V} = \frac{16\pi\zeta(3)}{c^3}\left(\frac{kT}{h}\right)^3 \propto T^3.$$

Heating a cavity does not merely give the existing photons more energy. It makes
more photons.

In [ ]:
# Cell 10 - Plot 9: the chemical potential, and how many photons a cavity holds

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: what a nonzero chemical potential would do to the occupancy
ax = axes[0]
energy_ratio = np.linspace(0.05, 6.0, 400)
for mu_ratio, color, style in [
    (0.0, "crimson", "-"),
    (-0.5, "royalblue", "--"),
    (-2.0, "seagreen", ":"),
]:
    ax.semilogy(
        energy_ratio,
        1.0 / np.expm1(energy_ratio - mu_ratio),
        color=color,
        ls=style,
        lw=2,
        label=rf"$\mu / kT$ = {mu_ratio}",
    )
ax.set_xlabel(r"$\varepsilon / kT$")
ax.set_ylabel("mean occupancy $n$")
ax.set_title(
    r"Photons live on the $\mu = 0$ curve, which diverges at $\varepsilon \to 0$"
)
ax.legend(fontsize=9, loc="upper right")
ax.grid(True, which="both", alpha=0.3)

# Right: photon number density really does scale as T^3


def photon_number_density(temperature: float) -> float:
    """Return photons per cubic meter in a cavity, by numerical integration."""

    def integrand(nu: float) -> float:
        """Photons per unit volume per unit frequency at this temperature."""
        return density_of_states(nu) / np.expm1(
            H_PLANCK * nu / (K_BOLTZMANN * temperature)
        )

    upper = 40.0 * K_BOLTZMANN * temperature / H_PLANCK  # x = 40 is deep in the tail
    value, _ = quad(integrand, 1e-6 * upper, upper, limit=200)
    return value


temperatures = np.logspace(0, 4, 40)
counted_density_vs_t = np.array([photon_number_density(t) for t in temperatures])
analytic_density = (
    16.0 * np.pi * zeta(3.0) * (K_BOLTZMANN * temperatures / (H_PLANCK * C_LIGHT)) ** 3
)

ax = axes[1]
ax.loglog(
    temperatures,
    counted_density_vs_t,
    color="royalblue",
    lw=2,
    label="numerical integration",
)
ax.loglog(
    temperatures,
    analytic_density,
    color="crimson",
    ls="--",
    lw=2,
    label=r"$16\pi\zeta(3)\,(kT/hc)^3$",
)
for label, temperature, color in [
    ("CMB", CMB_TEMPERATURE, "darkorange"),
    ("Sun", SUN_TEMPERATURE, "seagreen"),
]:
    ax.plot(
        temperature,
        photon_number_density(temperature),
        "o",
        color=color,
        ms=9,
        mec="k",
        zorder=5,
        label=f"{label} ({temperature:.6g} K)",
    )
ax.set_xlabel("$T$ (K)")
ax.set_ylabel("photons per m$^3$")
ax.set_title("Photon number is set by the temperature, not conserved")
ax.legend(fontsize=9, loc="upper left")
ax.grid(True, which="both", alpha=0.3)

fig.suptitle(r"Step 12: With No Particle-Number Constraint, $\mu = 0$")
fig.tight_layout()
plt.show()

for label, temperature in [("CMB", CMB_TEMPERATURE), ("Sun", SUN_TEMPERATURE)]:
    numeric = photon_number_density(temperature)
    exact = (
        16.0
        * np.pi
        * float(zeta(3.0))
        * (K_BOLTZMANN * temperature / (H_PLANCK * C_LIGHT)) ** 3
    )
    print(
        f"{label:>4} at {temperature:8.2f} K: {numeric:.6e} photons/m^3 "
        f"(formula {exact:.6e}, ratio {numeric / exact:.6f})"
    )
print()
print(
    f"doubling T multiplies the photon count by "
    f"{photon_number_density(600.0) / photon_number_density(300.0):.4f}  (expected 8)"
)

---
## Step 13. The average energy in one state

We now have both halves of the answer. Each photon in a state of frequency $\nu$
carries $h\nu$, and the mean number of photons in that state is
$\overline{n}(\nu)$, so the mean energy of one state is simply the product:

$$\boxed{\;\overline{\varepsilon}(\nu) = h\nu\,\overline{n}(\nu)
  = \frac{h\nu}{e^{h\nu/kT}-1}\;}$$

Look at the two limits, writing $x = h\nu/kT$.

**Low frequency, $x \ll 1$.** Expand $e^x - 1 \approx x$, and

$$\overline{\varepsilon} \approx \frac{h\nu}{h\nu/kT} = kT.$$

Every low-frequency mode carries $kT$, exactly what classical equipartition
predicts. Bose statistics reproduces classical physics where classical physics
worked.

**High frequency, $x \gg 1$.** Now $e^x - 1 \approx e^x$, and

$$\overline{\varepsilon} \approx h\nu\,e^{-h\nu/kT},$$

which collapses to nothing exponentially fast. This is the cure for the
ultraviolet catastrophe, and the reason is easy to state in words: a
high-frequency mode can only accept energy in a lump of size $h\nu$, and if that
lump costs far more than $kT$, the mode almost always ends up holding **zero**
photons. The classical calculation, which lets a mode absorb energy in arbitrarily
small amounts, has no way to see this.

In [ ]:
# Cell 11 - Plot 10: mean energy per state, quantum against classical


def mean_energy_per_state(
    frequency: float | np.ndarray, temperature: float
) -> np.ndarray:
    """Return the average energy in joules of one photon state at this frequency."""
    x = H_PLANCK * np.asarray(frequency, dtype=float) / (K_BOLTZMANN * temperature)
    return H_PLANCK * np.asarray(frequency, dtype=float) / np.expm1(x)


fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: absolute energy per state at the Sun's surface temperature
ax = axes[0]
spectrum = np.logspace(12, 15.5, 500)
ax.loglog(
    spectrum,
    mean_energy_per_state(spectrum, SUN_TEMPERATURE) / elementary_charge,
    color="royalblue",
    lw=2,
    label=r"Bose:  $h\nu / (e^{h\nu/kT}-1)$",
)
ax.axhline(
    K_BOLTZMANN * SUN_TEMPERATURE / elementary_charge,
    color="crimson",
    ls="--",
    lw=2,
    label=r"classical equipartition:  $kT$",
)
ax.axvline(
    K_BOLTZMANN * SUN_TEMPERATURE / H_PLANCK,
    color="gray",
    ls=":",
    lw=1.5,
    label=r"$h\nu = kT$",
)
ax.set_ylim(1e-6, 10.0)
ax.set_xlabel(r"$\nu$ (Hz)")
ax.set_ylabel("mean energy per state (eV)")
ax.set_title(
    f"At T = {SUN_TEMPERATURE:.0f} K the modes shut off above $h\\nu \\approx kT$"
)
ax.legend(fontsize=9, loc="lower left")
ax.grid(True, which="both", alpha=0.3)

# Right: the same statement in dimensionless form, with both limits drawn
ax = axes[1]
x = np.logspace(-2, 1.3, 400)
ax.loglog(
    x,
    x / np.expm1(x),
    color="royalblue",
    lw=2.5,
    label=r"$\overline{\varepsilon}/kT = x/(e^x-1)$",
)
ax.loglog(
    x,
    np.ones_like(x),
    color="crimson",
    ls="--",
    lw=1.6,
    label=r"low $x$: classical $kT$",
)
ax.loglog(
    x,
    x * np.exp(-x),
    color="seagreen",
    ls=":",
    lw=2,
    label=r"high $x$: Wien $h\nu e^{-h\nu/kT}$",
)
ax.set_ylim(1e-4, 3.0)
ax.set_xlabel(r"$x = h\nu / kT$")
ax.set_ylabel(r"$\overline{\varepsilon} / kT$")
ax.set_title("One curve covers every temperature")
ax.legend(fontsize=9, loc="lower left")
ax.grid(True, which="both", alpha=0.3)

fig.suptitle("Step 13: A Mode Too Expensive to Fill Stays Empty")
fig.tight_layout()
plt.show()

print(f"{'x = h nu / kT':>16}{'mean n':>16}{'eps / kT':>16}{'classical':>12}")
for x_value in [0.01, 0.1, 1.0, 5.0, 10.0, 20.0]:
    print(
        f"{x_value:16.2f}{bose_occupancy(x_value):16.6e}"
        f"{x_value / np.expm1(x_value):16.6e}{1.0:12.2f}"
    )

---
## Step 14. Multiply, and Planck's law falls out

Nothing is left to do but combine the two results.

Number of states per unit volume per unit frequency, from step 6:

$$\frac{g(\nu)}{V} = \frac{8\pi\nu^2}{c^3}.$$

Average energy per state, from step 13:

$$\overline{\varepsilon}(\nu) = \frac{h\nu}{e^{h\nu/kT}-1}.$$

Their product is the energy per unit volume per unit frequency:

$$u(\nu, T) = \frac{8\pi\nu^2}{c^3}\cdot\frac{h\nu}{e^{h\nu/kT}-1}$$

$$\boxed{\;u(\nu, T) = \frac{8\pi h\nu^3}{c^3}\,\frac{1}{e^{h\nu/kT}-1}\;}$$

That is Planck's law, obtained without a single classical oscillator, without
Maxwell's equations, and without any classical statistical mechanics. The whole
derivation was: count the cells, count the occupations, take a logarithm,
maximize.

The failed classical result is now visible as an approximation. Replacing
$\overline{\varepsilon}$ with the equipartition value $kT$ gives the
**Rayleigh-Jeans law**,

$$u_\text{RJ} = \frac{8\pi\nu^2}{c^3}kT,$$

which rises forever and integrates to infinite energy: the ultraviolet
catastrophe. Keeping only the $e^{-h\nu/kT}$ tail gives **Wien's law**, correct at
high frequency and wrong at low. Planck's formula is the one expression that
matches both ends, and Bose derived it from counting alone.

In [ ]:
# Cell 12 - Plot 11: Planck's law, with the two classical approximations


def planck_law(frequency: float | np.ndarray, temperature: float) -> np.ndarray:
    """Return the spectral energy density u(nu, T) in J m^-3 Hz^-1.

    Parameters
    ----------
    frequency : float | np.ndarray
        Frequency in Hz.
    temperature : float
        Cavity temperature in kelvin.

    Returns
    -------
    np.ndarray
        Energy per unit volume per unit frequency.
    """
    nu = np.asarray(frequency, dtype=float)
    return (
        density_of_states(nu)
        * H_PLANCK
        * nu
        / np.expm1(H_PLANCK * nu / (K_BOLTZMANN * temperature))
    )


def rayleigh_jeans(frequency: float | np.ndarray, temperature: float) -> np.ndarray:
    """Return the classical energy density, density of states times kT."""
    return (
        density_of_states(np.asarray(frequency, dtype=float))
        * K_BOLTZMANN
        * temperature
    )


def wien_law(frequency: float | np.ndarray, temperature: float) -> np.ndarray:
    """Return Wien's high-frequency approximation to the energy density."""
    nu = np.asarray(frequency, dtype=float)
    return (
        density_of_states(nu)
        * H_PLANCK
        * nu
        * np.exp(-H_PLANCK * nu / (K_BOLTZMANN * temperature))
    )


fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: the family of Planck curves, on linear axes
ax = axes[0]
linear_frequency = np.linspace(1e12, 2.0e15, 600)
for temperature, color in [
    (3000.0, "royalblue"),
    (4000.0, "seagreen"),
    (5000.0, "darkorange"),
    (SUN_TEMPERATURE, "crimson"),
]:
    ax.plot(
        linear_frequency / 1e14,
        planck_law(linear_frequency, temperature) * 1e14,
        color=color,
        lw=2,
        label=f"T = {temperature:.0f} K",
    )
ax.axvspan(
    VISIBLE_LOW / 1e14,
    VISIBLE_HIGH / 1e14,
    color="palegreen",
    alpha=0.6,
    label="visible",
)
ax.set_xlabel(r"$\nu$ ($10^{14}$ Hz)")
ax.set_ylabel(r"$u(\nu,T)$  (J m$^{-3}$ per $10^{14}$ Hz)")
ax.set_title("Planck's law at four temperatures")
ax.legend(fontsize=9, loc="upper right")
ax.grid(True, alpha=0.3)

# Right: how the classical limits fail, shown on log-log at one temperature
ax = axes[1]
log_frequency = np.logspace(12.5, 15.7, 500)
ax.loglog(
    log_frequency,
    planck_law(log_frequency, SUN_TEMPERATURE),
    color="crimson",
    lw=2.5,
    label="Planck (Bose counting)",
)
ax.loglog(
    log_frequency,
    rayleigh_jeans(log_frequency, SUN_TEMPERATURE),
    color="royalblue",
    ls="--",
    lw=1.8,
    label=r"Rayleigh-Jeans, $8\pi\nu^2 kT/c^3$",
)
ax.loglog(
    log_frequency,
    wien_law(log_frequency, SUN_TEMPERATURE),
    color="seagreen",
    ls=":",
    lw=2,
    label="Wien",
)
ax.annotate(
    "ultraviolet\ncatastrophe",
    xy=(1.0e15, float(rayleigh_jeans(1.0e15, SUN_TEMPERATURE))),
    xytext=(1.1e14, 1.0e-15),
    arrowprops={"arrowstyle": "->", "lw": 1.4},
    fontsize=10,
    ha="center",
)
ax.set_ylim(1e-22, 1e-13)
ax.set_xlabel(r"$\nu$ (Hz)")
ax.set_ylabel(r"$u(\nu,T)$  (J m$^{-3}$ Hz$^{-1}$)")
ax.set_title(f"T = {SUN_TEMPERATURE:.0f} K: the classical curve never turns over")
ax.legend(fontsize=9, loc="lower left")
ax.grid(True, which="both", alpha=0.3)

fig.suptitle("Step 14: Density of States Times Energy per State Is Planck's Law")
fig.tight_layout()
plt.show()

peak_frequency = linear_frequency[
    np.argmax(planck_law(linear_frequency, SUN_TEMPERATURE))
]
print(
    f"at T = {SUN_TEMPERATURE:.0f} K the curve of u(nu, T) peaks near "
    f"{peak_frequency:.3e} Hz"
)
print("(the familiar 500 nm solar peak belongs to the wavelength form of the")
print(" spectrum, which peaks somewhere else - see the next section)")
print()
print("Planck against the two classical limits at T = 5772 K:")
print(f"{'nu (Hz)':>12}{'x':>8}{'Planck':>14}{'Rayleigh-Jeans':>18}{'Wien':>14}")
for nu in [1e13, 1e14, 3.4e14, 1e15, 2e15]:
    print(
        f"{nu:12.1e}{H_PLANCK * nu / (K_BOLTZMANN * SUN_TEMPERATURE):8.3f}"
        f"{float(planck_law(nu, SUN_TEMPERATURE)):14.4e}"
        f"{float(rayleigh_jeans(nu, SUN_TEMPERATURE)):18.4e}"
        f"{float(wien_law(nu, SUN_TEMPERATURE)):14.4e}"
    )

### Does the result actually reproduce the measured constants?

A derivation this abstract deserves a hard check against the laboratory. Two
famous empirical laws pre-date Planck and follow from integrating his formula, so
they are exactly the right test.

**Stefan-Boltzmann.** Integrating $u(\nu,T)$ over all frequencies, with
$x = h\nu/kT$ and $\int_0^\infty x^3/(e^x-1)\,dx = \pi^4/15$, gives

$$U(T) = \int_0^\infty u(\nu,T)\,d\nu = \frac{4\sigma}{c}T^4,
\qquad
\sigma = \frac{2\pi^5 k^4}{15 h^3 c^2}.$$

The constant $\sigma$ was measured by Stefan in 1879 from experimental data, long
before anybody knew what $h$ was.

**Wien displacement.** Setting $du_\lambda/d\lambda = 0$ leads to
$x = 5(1-e^{-x})$, whose root is $x \approx 4.965$, and therefore

$$\lambda_\text{max}T = \frac{hc}{4.965\,k} = b \approx 2.898\times10^{-3}\ \text{m K}.$$

The same exercise on $u(\nu,T)$ gives a different root, $x \approx 2.821$, because
a spectrum per unit frequency and a spectrum per unit wavelength peak at different
places. That is not a contradiction, it is what happens when a density is
reparameterized.

The cell below recomputes both from the derivation and compares them with the
CODATA values that SciPy carries.

In [ ]:
# Cell 13 - Plot 12: checking the derivation against measured constants


def total_energy_density(temperature: float) -> float:
    """Return the energy per cubic meter in a cavity, integrated over frequency."""
    upper = 60.0 * K_BOLTZMANN * temperature / H_PLANCK
    value, _ = quad(lambda nu: planck_law(nu, temperature), 0.0, upper, limit=400)
    return value


# Stefan-Boltzmann constant, derived from our own integral
check_temperature = 1000.0
sigma_derived = (
    total_energy_density(check_temperature) * C_LIGHT / (4.0 * check_temperature**4)
)

# Wien displacement, from the peak of the wavelength form of the spectrum
wien_root_lambda = root_scalar(
    lambda x: x - 5.0 * (1.0 - np.exp(-x)), bracket=(1.0, 10.0)
).root
wien_root_nu = root_scalar(
    lambda x: x - 3.0 * (1.0 - np.exp(-x)), bracket=(1.0, 10.0)
).root
wien_constant = H_PLANCK * C_LIGHT / (wien_root_lambda * K_BOLTZMANN)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: the T^4 law, computed point by point
ax = axes[0]
check_temps = np.logspace(0.5, 4, 30)
integrated = np.array([total_energy_density(t) for t in check_temps])
ax.loglog(
    check_temps,
    integrated,
    "o",
    color="royalblue",
    ms=6,
    label="numerical integral of $u(\\nu,T)$",
)
ax.loglog(
    check_temps,
    4.0 * Stefan_Boltzmann * check_temps**4 / C_LIGHT,
    color="crimson",
    ls="--",
    lw=2,
    label=r"$4\sigma T^4/c$ with CODATA $\sigma$",
)
ax.set_xlabel("$T$ (K)")
ax.set_ylabel("energy density (J m$^{-3}$)")
ax.set_title("Stefan-Boltzmann: the integral really does go as $T^4$")
ax.legend(fontsize=9, loc="upper left")
ax.grid(True, which="both", alpha=0.3)

# Right: the cosmic microwave background, the best blackbody ever measured
ax = axes[1]
cmb_frequency = np.linspace(1e9, 6e11, 500)
ax.plot(
    cmb_frequency / 1e9,
    planck_law(cmb_frequency, CMB_TEMPERATURE) * 1e9,
    color="darkorange",
    lw=2.5,
    label=f"Planck at T = {CMB_TEMPERATURE} K",
)
cmb_peak = wien_root_nu * K_BOLTZMANN * CMB_TEMPERATURE / H_PLANCK
ax.axvline(
    cmb_peak / 1e9, color="k", ls=":", lw=1.5, label=f"peak at {cmb_peak / 1e9:.1f} GHz"
)
ax.set_xlabel(r"$\nu$ (GHz)")
ax.set_ylabel(r"$u(\nu,T)$  (J m$^{-3}$ per GHz)")
ax.set_title("The CMB, a blackbody filling the observable universe")
ax.legend(fontsize=9, loc="upper right")
ax.grid(True, alpha=0.3)

fig.suptitle("Bose's Counting Argument Predicts Constants Measured Decades Earlier")
fig.tight_layout()
plt.show()

print("Stefan-Boltzmann sigma")
print(f"  from this notebook  = {sigma_derived:.9e} W m^-2 K^-4")
print(
    f"  closed form         = "
    f"{2 * np.pi**5 * K_BOLTZMANN**4 / (15 * H_PLANCK**3 * C_LIGHT**2):.9e}"
)
print(f"  CODATA value        = {Stefan_Boltzmann:.9e}")
print(f"  relative difference = {sigma_derived / Stefan_Boltzmann - 1:.3e}")
print()
print("Wien displacement constant b")
print(f"  root of x = 5(1 - e^-x) = {wien_root_lambda:.9f}")
print(f"  b = hc / (x k)          = {wien_constant:.9e} m K")
print(f"  CODATA value            = {Wien:.9e} m K")
print(f"  relative difference     = {wien_constant / Wien - 1:.3e}")
print()
print(
    f"peak of the frequency spectrum sits at x = {wien_root_nu:.6f}, not {wien_root_lambda:.6f}"
)
print(
    f"  Sun ({SUN_TEMPERATURE:.0f} K): peak at "
    f"{wien_root_nu * K_BOLTZMANN * SUN_TEMPERATURE / H_PLANCK:.4e} Hz"
)
print(
    f"  CMB ({CMB_TEMPERATURE} K): peak at "
    f"{wien_root_nu * K_BOLTZMANN * CMB_TEMPERATURE / H_PLANCK / 1e9:.2f} GHz"
)

---
## The whole argument on one page

| Step | Statement |
| --- | --- |
| 1 | A photon is a particle with $\varepsilon = h\nu$ and $p = h\nu/c$ |
| 2 | One quantum state occupies a phase-space cell of volume $h^3$ |
| 3-5 | A momentum shell holds $g(p)\,dp = 8\pi V p^2 dp / h^3$ states, the 8 from two polarizations |
| 6 | In frequency, $g(\nu)\,d\nu = 8\pi V\nu^2 d\nu / c^3$, with $h$ canceling out |
| 7-8 | Indistinguishable photons in $g$ states give $W = (N+g-1)!\,/\,[N!\,(g-1)!]$ |
| 9 | $S = k\ln W$, and Stirling turns it into $k\sum[(N+g)\ln(N+g) - N\ln N - g\ln g]$ |
| 10-11 | Maximizing $S - \lambda E$ gives $\overline{n} = 1/(e^{h\nu/kT}-1)$ |
| 12 | Photon number is not conserved, so $\mu = 0$ |
| 13 | Mean energy per state is $h\nu\,\overline{n} = h\nu/(e^{h\nu/kT}-1)$ |
| 14 | Multiply by the density of states: $u(\nu,T) = \dfrac{8\pi h\nu^3}{c^3}\dfrac{1}{e^{h\nu/kT}-1}$ |

Every quantum feature of the result traces back to **one** decision, made in step
7: record how many photons are in each state, never which ones. Count with labels
instead and $W$ becomes $g^N$, the entropy becomes $k N \ln g$ plus terms without
the crucial $(N+g)\ln(N+g)$, the occupancy becomes $e^{-h\nu/kT}$, and the
spectrum is Wien's, not Planck's.

### What Einstein did next

Bose stopped at photons. Einstein applied the identical counting to a gas of
massive atoms with $\varepsilon = p^2/2m$, where the particle number **is**
conserved. That extra constraint brings back the second Lagrange multiplier and
therefore the chemical potential,

$$\overline{n}(\varepsilon) = \frac{1}{e^{(\varepsilon-\mu)/kT}-1},$$

and $\mu$ must adjust itself to hold $N$ fixed. Einstein noticed that $\mu$ cannot
exceed the ground-state energy, so below a critical temperature the excited states
simply cannot hold all the atoms. The surplus collects in the single lowest state:
**Bose-Einstein condensation**, predicted in 1925 and produced in a rubidium vapor
at 170 nK by Cornell and Wieman in 1995.

The deep innovation was never the phase-space cells, which were in the air at the
time. It was the decision to treat identical particles as genuinely identical.